# Lab 4 – Logical Reasoning for Planning

**Name:** Mehta Prem Kalpesh 

**ID:** 2024ADPS0851G

## Task 0:

**(a) Initial state**

I = {At(Robot, A), At(Package, A)}

**(b) Goal**

G = {At(Package, C)}

**(c) Available actions**

- **Move(X, Y)** for each connected pair of locations: Move(A, B), Move(B, A), Move(B, C), Move(C, B)
- **PickUp(Package, L)** for L in {A, B, C}
- **Drop(Package, L)** for L in {A, B, C}

**(d) Preconditions and effects**

| Action | Preconditions | Effects |
|---|---|---|
| Move(X, Y) | At(Robot, X) | ¬At(Robot, X), At(Robot, Y) |
| PickUp(Package, L) | At(Robot, L), At(Package, L) | ¬At(Package, L), Holding(Package) |
| Drop(Package, L) | At(Robot, L), Holding(Package) | ¬Holding(Package), At(Package, L) |

(Move is only defined between connected locations: A–B and B–C. There is no direct A–C move.)

### Which actions are initially applicable?

Starting state: I = {At(Robot, A), At(Package, A)}

**1. Is PickUp(Package, A) applicable? Yes.**

Preconditions: At(Robot, A) and At(Package, A).
- At(Robot, A) is in I ✓
- At(Package, A) is in I ✓

All preconditions are satisfied, so I ⊨ Preconditions(PickUp(Package, A)) and the action is applicable.

**2. Is Drop(Package, C) applicable? No.**

Preconditions: At(Robot, C) and Holding(Package).
- At(Robot, C) is not in I ✗ (the robot is at A)
- Holding(Package) is not in I ✗ (the robot has not picked up the package)

Both preconditions fail, so I does not entail Preconditions(Drop(Package, C)) and the action is not applicable.

## Task 1: Construct a Plan by Hand

**Plan:** PickUp(Package, A) → Move(A, B) → Move(B, C) → Drop(Package, C)

| State | Action applied | Facts |
|---|---|---|
| S0 | (initial) | At(Robot, A), At(Package, A) |
| S1 | PickUp(Package, A) | At(Robot, A), Holding(Package) |
| S2 | Move(A, B) | At(Robot, B), Holding(Package) |
| S3 | Move(B, C) | At(Robot, C), Holding(Package) |
| S4 | Drop(Package, C) | At(Robot, C), At(Package, C) |

## Task 2: Ask an LLM to Implement the Planner

## Prompt:

I want to implement a simple planning agent in Python 3 (standard library only, single file).

## Planner specification
Represent a state as a set of logical propositions (e.g. a frozenset of strings such as "At(Robot,A)").
Each action should contain:
- a name;
- positive preconditions;
- negative preconditions;
- positive effects;
- negative effects.

An action is applicable if all of its positive preconditions are in the current state and none of its negative preconditions are in the current state.
When an action is applied:
1. remove its negative effects from the state;
2. add its positive effects to the state.

Use breadth-first search to find a sequence of actions that achieves a specified goal (the goal is satisfied when every goal proposition is in the state).

The program should also:
- detect when no plan exists and print "No plan found";
- print the resulting sequence of actions;
- print the states reached after each action (starting with the initial state).

## Design requirements
- Keep the problem definition (initial state, goal, list of actions) separate from the planner code, so I can change the problem (e.g. remove an action or add a new one) without touching the search.
- The planner should be a function that takes (initial_state, goal, actions) and returns a list of actions, or None if no plan exists.
- Avoid revisiting states already explored.
- Include a main section that runs the planner on the problem below.

## Problem: warehouse robot
A robot must transport a package between three locations: A, B, C.
The only connections are A–B and B–C (in both directions). There is no direct A–C connection.

Initial state: {At(Robot,A), At(Package,A)}
Goal: {At(Package,C)}

Action schemas (ground them over the locations; Move only for connected pairs):

| Action | Preconditions | Effects |
|---|---|---|
| Move(X,Y) | At(Robot,X) | ¬At(Robot,X), At(Robot,Y) |
| PickUp(Package,L) | At(Robot,L), At(Package,L) | ¬At(Package,L), Holding(Package) |
| Drop(Package,L) | At(Robot,L), Holding(Package) | ¬Holding(Package), At(Package,L) |

Explain the implementation and identify any assumptions you make.

In [ ]:
from collections import deque

class Action:
    """Represents a grounded action with preconditions and effects."""
    def __init__(self, name, pos_pre, neg_pre, pos_eff, neg_eff):
        self.name = name
        # Using frozenset for efficient set operations and immutability
        self.pos_pre = frozenset(pos_pre)
        self.neg_pre = frozenset(neg_pre)
        self.pos_eff = frozenset(pos_eff)
        self.neg_eff = frozenset(neg_eff)

    def is_applicable(self, state):
        """
        An action is applicable if all positive preconditions are in the state,
        and no negative preconditions are in the state.
        """
        return self.pos_pre.issubset(state) and self.neg_pre.isdisjoint(state)

    def apply(self, state):
        """
        Applies the action by removing negative effects and adding positive effects.
        """
        return (state - self.neg_eff) | self.pos_eff


def bfs_plan(initial_state, goal, actions):
    """
    Finds the shortest sequence of actions to reach a state that satisfies the goal.
    Returns a list of Action objects, or None if no plan exists.
    """
    # Check if the initial state already satisfies the goal
    if goal.issubset(initial_state):
        return []

    # Queue stores tuples of (current_state, path_of_actions)
    queue = deque([(initial_state, [])])
    
    # Track visited states to avoid cycles. 
    # Works safely because states are frozensets (which are hashable).
    visited = {initial_state}

    while queue:
        current_state, path = queue.popleft()

        for action in actions:
            if action.is_applicable(current_state):
                new_state = action.apply(current_state)
                
                if new_state not in visited:
                    new_path = path + [action]
                    
                    # Check if the new state satisfies our goal
                    if goal.issubset(new_state):
                        return new_path
                    
                    visited.add(new_state)
                    queue.append((new_state, new_path))
                    
    return None  # Queue exhausted without finding a goal


def print_execution(initial_state, plan):
    """Executes the plan and prints the state progression."""
    if plan is None:
        print("No plan found.")
        return

    print("=== Plan Execution ===")
    print(f"Step 0 (Initial state): {sorted(list(initial_state))}")
    
    current_state = initial_state
    for i, action in enumerate(plan, 1):
        current_state = action.apply(current_state)
        print(f"\nAction {i}: {action.name}")
        print(f"Step {i} (State): {sorted(list(current_state))}")


def main():
    # 1. Define Initial State and Goal
    initial_state = frozenset({"At(Robot,A)", "At(Package,A)"})
    goal = frozenset({"At(Package,C)"})

    # 2. Define Grounded Actions
    # We manually ground the schemas (Move, PickUp, Drop) over locations A, B, C.
    actions = [
        # Move Actions (Connections: A-B, B-C in both directions)
        Action("Move(A,B)", pos_pre={"At(Robot,A)"}, neg_pre=set(), 
               pos_eff={"At(Robot,B)"}, neg_eff={"At(Robot,A)"}),
        Action("Move(B,A)", pos_pre={"At(Robot,B)"}, neg_pre=set(), 
               pos_eff={"At(Robot,A)"}, neg_eff={"At(Robot,B)"}),
        Action("Move(B,C)", pos_pre={"At(Robot,B)"}, neg_pre=set(), 
               pos_eff={"At(Robot,C)"}, neg_eff={"At(Robot,B)"}),
        Action("Move(C,B)", pos_pre={"At(Robot,C)"}, neg_pre=set(), 
               pos_eff={"At(Robot,B)"}, neg_eff={"At(Robot,C)"}),
        
        # PickUp Actions
        Action("PickUp(Package,A)", pos_pre={"At(Robot,A)", "At(Package,A)"}, neg_pre=set(), 
               pos_eff={"Holding(Package)"}, neg_eff={"At(Package,A)"}),
        Action("PickUp(Package,B)", pos_pre={"At(Robot,B)", "At(Package,B)"}, neg_pre=set(), 
               pos_eff={"Holding(Package)"}, neg_eff={"At(Package,B)"}),
        Action("PickUp(Package,C)", pos_pre={"At(Robot,C)", "At(Package,C)"}, neg_pre=set(), 
               pos_eff={"Holding(Package)"}, neg_eff={"At(Package,C)"}),
        
        # Drop Actions
        Action("Drop(Package,A)", pos_pre={"At(Robot,A)", "Holding(Package)"}, neg_pre=set(), 
               pos_eff={"At(Package,A)"}, neg_eff={"Holding(Package)"}),
        Action("Drop(Package,B)", pos_pre={"At(Robot,B)", "Holding(Package)"}, neg_pre=set(), 
               pos_eff={"At(Package,B)"}, neg_eff={"Holding(Package)"}),
        Action("Drop(Package,C)", pos_pre={"At(Robot,C)", "Holding(Package)"}, neg_pre=set(), 
               pos_eff={"At(Package,C)"}, neg_eff={"Holding(Package)"})
    ]

    # 3. Run the Planner
    plan = bfs_plan(initial_state, goal, actions)

    # 4. Output the results
    print_execution(initial_state, plan)


if __name__ == "__main__":
    main()

No plan found.


## Task 2: Think About It

| Idea from the specification | Where it appears in the program | What it does |
|---|---|---|
| **Preconditions** → When is an action applicable? | `Action.is_applicable(state)` (line 18), called from `bfs_plan` (line 47) | Returns `True` only if every positive precondition is in the state (`pos_pre.issubset(state)`) and no negative precondition is in the state (`neg_pre.isdisjoint(state)`). This is the check S ⊨ Preconditions(a). |
| **Effects** → How does the state change? | `Action.apply(state)` (line 24) | Returns `(state - neg_eff) \| pos_eff`: the negative effects are removed first, then the positive effects are added, which is the order the specification gave. The original state is not modified; a new state is returned. |
| **Goal** → When does planning terminate? | The goal test `goal.issubset(new_state)` (line 54), plus `goal.issubset(initial_state)` (line 33) | The goal itself is the data `goal = frozenset({"At(Package,C)"})` in `main()`. Planning terminates when a newly generated state contains every goal proposition, and the path that led to it is returned. If the initial state already satisfies the goal, an empty plan is returned. If the queue empties without a goal state, the function returns `None`, which is reported as "No plan found". |
| **BFS** → How are alternative plans explored? | `bfs_plan` (lines 27–60): a FIFO `deque` with `popleft()` (line 44), a `for` loop over all actions (line 46), and the `visited` set (line 41) | Each applicable action produces a successor state that is added to the back of the queue together with the action sequence that reached it. Because the queue is first-in, first-out, all plans of length k are explored before any plan of length k+1, so the first plan found is a shortest one. The `visited` set stops the search from revisiting states, for example in the loop A → B → A → B. |

**Assumptions visible in the generated code**
- A state is a `frozenset` of proposition strings, and a proposition that is absent from the state is treated as false (closed-world assumption).
- The actions were grounded by hand over the locations A, B and C, with `Move` only on the connected pairs A–B and B–C.
- Every action has the same cost, so BFS finds a plan with the fewest actions.

## Task 3: Test the Generated Planner

### Test A: Solvable problem
- **Initial state:** {At(Robot,A), At(Package,A)}
- **Goal:** {At(Package,C)}
- **Plan found?** Yes
- **Plan:** PickUp(Package,A) → Move(A,B) → Move(B,C) → Drop(Package,C)
- **Valid?** Yes. The validator confirmed that every action's preconditions held in the state before it. The final state is {At(Robot,C), At(Package,C)}, which satisfies the goal.
- The plan has 4 actions, which is the minimum possible (one pick-up, two moves, one drop). It is identical to my hand-built plan from Task 1, as expected from BFS.

### Test B: Impossible problem
- **Change:** all PickUp actions removed.
- **Initial state:** {At(Robot,A), At(Package,A)}
- **Goal:** {At(Package,C)}
- **Plan found?** No. `bfs_plan` returned `None`, which the program reports as "No plan found".
- **Valid?** Not applicable, since no plan was invented. Without PickUp, the package can never leave A.

### Test C: Irrelevant actions
- **C1 (control):** only Move actions, goal {At(Robot,C)}. Plan found: Move(A,B) → Move(B,C). Valid. This shows the robot can reach C.
- **C2:** same Move-only actions, goal {At(Package,C)}. No plan found. The robot reaches C in C1, but the planner does not treat that as the package being at C.
- **C3:** the full problem plus an extra robot-only action, Wander(A), which does not move the package. The plan is the same 4 actions as in Test A, Wander is not used, and the plan is valid.

### Summary
| Test | Plan found | Plan valid |
|---|---|---|
| A: solvable | Yes (4 actions) | Yes |
| B: PickUp removed | No | n/a (correct) |
| C1: robot to C | Yes (2 actions) | Yes |
| C2: package to C, Move-only | No | n/a (correct) |
| C3: extra Wander action | Yes (4 actions) | Yes |
| Extra: goal already true | Yes (empty plan) | Yes |
| Extra: C unreachable | No | n/a (correct) |

No bugs were found in the generated planner.